# Two Traders with Private Markov-Chain Signals

The game of `two_traders.ipynb` with each signal replaced by a two-state Markov chain $\alpha^i_t \in \{-1, 1\}$
switching at rate $\kappa = 1$, private to trader $i$. The conditional expectation of the other signal is the
Wonham filter, nonlinear in the observed path, so the linear-strategy solver does not apply.

At $\lambda = 0$ the traders decouple. The exact solution combines the discrete Bayes filter of the other chain with
a backward recursion for each trader, and is the reference.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import MarkovTraders
game = MarkovTraders(lam=0.0)

## 1. Learned and exact equilibria at $\lambda = 0$

The learned method is as in `two_traders.ipynb`, at the same small budget of 5,000 paths, with damping 0.5 and a
control variate that also includes the compensated chain increments $\alpha^i_{t_{k+1}} - e^{-2\kappa\Delta t}\alpha^i_{t_k}$.

In [ ]:
solver = Solver(game, damping=0.5, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(8, average_from=3, reference=exact)
print(f'relative L2 error: {relative_l2(learned[0], exact[0]):.3f}, {relative_l2(learned[1], exact[1]):.3f}')

In [ ]:
plot_controls(learned, exact, game.dt, ['trader 1', 'trader 2'], ylabel=r'$\nu_t$')

## 2. Private information at $\lambda = 1$

At $\lambda = 1$ no exact solution is known. The first order condition residual measures how far each control is
from the regression of its own target, fitted afresh on new paths.

In [ ]:
game = MarkovTraders(lam=1.0)
solver = Solver(game, damping=0.6, steps=300, batch=1000, n_paths=5000)
learned, _ = solver.run(6, average_from=3, log=lambda message: None)
print('first order condition residuals:', [round(first_order_residual(solver, i), 3) for i in range(2)])
print('mean |nu|:', [round(float(c.abs().mean()), 4) for c in learned])